<a href="https://colab.research.google.com/github/Vanessa-Rodrigues-156/TISD/blob/main/Sthira_colab_v5/Sthira_test_mcp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Complete Mental Health Chatbot Pipeline with Enhanced Contextual Memory,
Modular MCP, and Safety Mechanism

This script performs:
1. Dataset Processing:
   - Downloads (if needed), cleans, and merges datasets.
2. Login & System Setup:
   - Logs into Hugging Face and prints system information.
3. Model Loading & Fine-Tuning:
   - Loads Mistral-7B with BitsAndBytes and applies LoRA fine-tuning.
4. Training Loop:
   - Processes the merged dataset in chunks and trains the model.
5. Saving & Ollama Modelfile:
   - Saves the final model and writes an Ollama modelfile.
6. Test Script with Enhanced Contextual Memory:
   - Implements a modular MCP (Mental Health Chatbot Prompt) defined separately.
   - Uses conversation history (enhanced contextual memory) to update context dynamically.
   - Incorporates a safety mechanism that appends critical help warnings if needed.
7. Final Checks & Download Setup:
   - Performs a quick local test and (in Colab) creates a ZIP archive for download.
"""


In [ ]:


# %% [markdown]
# ## Cell 1: Dataset Processing & Cleaning
# This cell downloads (optional), loads, cleans, and merges datasets.
#####################################
# 1. DATASET PROCESSING & CLEANING  #
#####################################

import os
import re
import glob
import pandas as pd
from bs4 import BeautifulSoup

print("=== DATASET PROCESSING ===")

# --- Kaggle Dataset Download (optional) ---
try:
    from kaggle.api.kaggle_api_extended import KaggleApi
    os.environ['KAGGLE_USERNAME'] = 'savonamendes'  # Replace with your Kaggle username
    os.environ['KAGGLE_KEY'] = '530094abea5fb5d47a7a4411f5c47efe'  # Replace with your Kaggle API key
    api = KaggleApi()
    api.authenticate()
    print("Downloading dataset from Kaggle...")
    api.dataset_download_files('neelghoshal/reddit-mental-health-data', path='.', unzip=True)
    files_list = glob.glob('*.csv')
    print("Available CSV files:", files_list)
except Exception as e:
    print("Kaggle download error:", str(e))

# --- Load & Clean Dataset ---
try:
    df = pd.read_csv('data_to_be_cleansed.csv')  # Adjust filename if needed
    print("Loaded dataset 'data_to_be_cleansed.csv'")
except Exception as e:
    print("Error loading CSV file:", str(e))

df_clean = df.copy()
if 'Unnamed: 0' in df_clean.columns:
    df_clean = df_clean.drop('Unnamed: 0', axis=1)

print("Missing values before cleaning:")
print(df_clean.isnull().sum())

# Drop rows with missing 'title' and 'text'
df_clean = df_clean.dropna(subset=['title', 'text'], how='all')

def clean_text(text):
    if isinstance(text, str):
        text = text.lower()
        text = BeautifulSoup(text, "html.parser").get_text()
        text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
        text = re.sub(r'\[.*?\]|\(.*?\)', '', text)
        text = re.sub(r'[^a-zA-Z\s]', '', text)
        text = ' '.join(text.split())
        return text
    return text

print("Cleaning text data...")
df_clean['text'] = df_clean['text'].apply(clean_text)
df_clean['title'] = df_clean['title'].apply(clean_text)

df_clean = df_clean[df_clean['text'].str.strip() != '']
df_clean = df_clean[df_clean['title'].str.strip() != '']
df_clean = df_clean.reset_index(drop=True)

print("Dataset shape before cleaning:", df.shape)
print("Dataset shape after cleaning:", df_clean.shape)
print("Sample of cleaned data:")
print(df_clean.head(2))

df_clean['text_length'] = df_clean['text'].str.len()
df_clean['title_length'] = df_clean['title'].str.len()
print("Text length statistics:")
print(df_clean[['text_length', 'title_length']].describe())

df_clean.to_csv('reddit_mental_health_cleaned.csv', index=False)
print("Cleaned dataset saved as 'reddit_mental_health_cleaned.csv'")

# --- Merge Multiple Datasets (if applicable) ---
files_to_merge = [
    "/content/TISD DATASET CSV - LD DA 1.csv",
    "/content/TISD DATASET CSV - LD EL1.csv",
    "/content/TISD DATASET CSV - LD PF1.csv",
    "/content/TISD DATASET CSV - LD TS 1.csv",
    "/content/reddit_mental_health_cleaned.csv",
    "/content/reddit_mental_health_cleaned.csv"
]
try:
    merged_df = pd.concat([pd.read_csv(f) for f in files_to_merge], ignore_index=True)
    merged_df.to_csv("merged_dataset.csv", index=False)
    print("Merged dataset saved as 'merged_dataset.csv'")
except Exception as e:
    print("Error merging datasets:", str(e))

In [ ]:
# %% [markdown]
# ## Cell 2: Login, System Setup & Model Loading
# This cell logs into Hugging Face, displays system info, and loads the model with LoRA and BitsAndBytes configuration.

from huggingface_hub import login
login()  # Login to Hugging Face; you'll be prompted to enter your token.

import torch
from datetime import datetime
print(f"Current Date and Time (UTC): {datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')}")
print("Current User's Login: Vanessa@dev")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Available GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, BitsAndBytesConfig, Trainer, DefaultDataCollator
from peft import LoraConfig, get_peft_model

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    llm_int8_enable_fp32_cpu_offload=True
)

print("Loading model...")
try:
    model_name = "mistralai/Mistral-7B-Instruct-v0.3"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True
    )

    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
except Exception as e:
    print(f"Error loading model: {str(e)}")
    raise e

In [ ]:
# %% [markdown]
# ## Cell 3: Training Loop
# This cell processes the merged dataset in chunks, tokenizes, and trains the model.

df = pd.read_csv('merged_dataset.csv')
df['title'] = df['title'].fillna('')
df['text'] = df['text'].fillna('')
df = df.dropna(subset=['title', 'text'], how='all')

def format_text(row):
    try:
        title = str(row['title']).strip()
        text = str(row['text']).strip()
        return f"[INST]Title: {title}\nContext: Provide empathetic advice or support based on this title.[/INST]\n{text[:1000]}</s>"
    except Exception as e:
        print(f"Error processing row: {e}")
        return "[INST]Error processing text[/INST]"

def prepare_inputs(batch):
    tokenized = tokenizer(
        batch['text'],
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors=None
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

chunk_size = 1000
total_chunks = len(df) // chunk_size + 1
output_path = "./mistral_mental_health"
os.makedirs(output_path, exist_ok=True)

for chunk_id in range(total_chunks):
    start_idx = chunk_id * chunk_size
    end_idx = min((chunk_id + 1) * chunk_size, len(df))
    chunk_df = df.iloc[start_idx:end_idx].copy()
    print(f"\nProcessing chunk {chunk_id + 1}/{total_chunks} (Rows: {len(chunk_df)})")
    chunk_df['text'] = chunk_df.apply(format_text, axis=1)

    from datasets import Dataset
    dataset = Dataset.from_pandas(chunk_df[['text']])
    tokenized_dataset = dataset.map(
        prepare_inputs,
        batched=True,
        remove_columns=dataset.column_names,
        desc="Tokenizing dataset"
    )

    print(f"Training chunk {chunk_id + 1}/{total_chunks}...")
    training_args = TrainingArguments(
        output_dir=f"./results_chunk_{chunk_id}",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        num_train_epochs=1,
        save_steps=100,
        save_total_limit=2,
        logging_steps=10,
        learning_rate=2e-4,
        max_grad_norm=0.3,
        gradient_checkpointing=True,
        fp16=True,
        optim="paged_adamw_32bit",
        warmup_ratio=0.03,
        weight_decay=0.01,
        report_to="tensorboard",
        remove_unused_columns=False,
        prediction_loss_only=True
    )

    model.train()
    for name, param in model.named_parameters():
        if param.dtype in [torch.float16, torch.float32, torch.bfloat16, torch.float64]:
            param.requires_grad = True

    from transformers import Trainer, DefaultDataCollator
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset,
        data_collator=DefaultDataCollator(),
        tokenizer=tokenizer
    )

    try:
        trainer.train()
        checkpoint_path = f"{output_path}/checkpoint-chunk-{chunk_id}"
        model.save_pretrained(checkpoint_path)
        print(f"Saved checkpoint for chunk {chunk_id + 1}")
    except Exception as e:
        print(f"Error during training chunk {chunk_id + 1}: {str(e)}")
        continue

    del dataset, tokenized_dataset
    torch.cuda.empty_cache()

print("Training complete. Saving final model...")
model.save_pretrained(output_path)
tokenizer.save_pretrained(output_path)

In [ ]:
# %% [markdown]
# ## Cell 4: Saving Ollama Modelfile
# This cell writes an Ollama Modelfile for later integration.

modelfile_content = """
FROM mistral
PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER stop "[INST]"
PARAMETER stop "[/INST]"

SYSTEM You are an AI trained on mental health discussions from Reddit. Respond with empathy and understanding.

LICENSE Apache 2.0
"""
with open(f"{output_path}/Modelfile", "w") as f:
    f.write(modelfile_content)
print(f"Model and Modelfile saved at {output_path}")

if torch.cuda.is_available():
    print("\nFinal GPU Memory Status:")
    print(f"Current Usage: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"Peak Usage: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

In [ ]:

# %% [markdown]
# ## Cell 5: Create Modular MCP and Contextual Memory Modules
# The following cells write separate module files for the MCP logic and contextual memory.

# %% [code]
%%writefile mcp_module.py
"""
Module: mcp_module.py

Defines the modular MCP template, safety check, and context generation function.
"""

MCP_TEMPLATE = r"""[ SYSTEM INSTRUCTIONS ]
You are a mental health chatbot designed to provide empathetic, supportive, and non-directive responses.
CONTEXT:
{
    "SessionID": "{session_id}",
    "UserEmotion": "{user_emotion}",
    "ConversationSummary": "{conversation_summary}",
    "EmotionalTrend": "{emotional_trend}",
    "BotRole": "Supportive friend",
    "BotPersona": "Empathetic, casual",
    "ConversationGoal": "Reassurance and validation",
    "UserStylePref": "Casual, friendly"
}
POLICIES:
1. EmpathyFirstPolicy: Always start by acknowledging the user's feelings (e.g., "I'm sorry you're feeling {user_emotion}").
2. NonDirectiveTonePolicy: Use gentle, suggestive language.
3. PersonaAdaptationPolicy: Maintain a casual, friendly tone.
4. ShortSummaryExpansion: Encourage the user to elaborate if needed.
5. PositiveReinforcementPolicy: Include phrases like "You're doing your best" or "You're not alone."
CUSTOM_MCPS: {custom_mcps}
ADDITIONAL GUIDELINES:
- If the emotional trend is worsening, gently remind the user that help is available.
{SAFETY_MESSAGE}
Please provide your response below.
"""

def check_safety(prompt: str) -> str:
    """
    Checks the prompt for critical keywords indicating a high-risk scenario.
    Returns a safety message if found.
    """
    critical_keywords = ["self harm", "suicide", "kill myself", "end my life", "overdose", "harm myself"]
    lower_prompt = prompt.lower()
    for keyword in critical_keywords:
        if keyword in lower_prompt:
            return (
                "SAFETY_MESSAGE: It appears you may be in critical distress. "
                "If you feel unsafe or are in immediate danger, please call your local emergency services immediately. "
                "Consider reaching out to a trusted person or a crisis support service."
            )
    return ""

def generate_mcp_context(session_id: str, user_emotion: str, conversation_summary: str,
                         emotional_trend: str, custom_mcps: str, prompt: str) -> str:
    """
    Populates the MCP template with dynamic values and a safety message.
    """
    safety_msg = check_safety(prompt)
    mcp_context = MCP_TEMPLATE.format(
        session_id=session_id,
        user_emotion=user_emotion,
        conversation_summary=conversation_summary,
        emotional_trend=emotional_trend,
        custom_mcps=custom_mcps,
        SAFETY_MESSAGE=safety_msg
    )
    return mcp_context


In [ ]:
# %% [markdown]
# ## Cell 6: Contextual Memory Module
# This module implements functions to manage multi-turn conversation memory.

# %% [code]
%%writefile context_memory.py
"""
Module: context_memory.py

Provides functions to update, retrieve, and reset conversation history.
"""

# Global list to store conversation turns.
conversation_history = []

def update_conversation_history(role: str, text: str):
    """
    Append a conversation turn to the history. Role is "User" or "Bot".
    """
    conversation_history.append(f"{role}: {text}")

def get_conversation_summary() -> str:
    """
    Generate a simple summary from the last several turns.
    """
    return " | ".join(conversation_history[-6:])

def reset_conversation_history():
    """
    Clear the conversation history.
    """
    conversation_history.clear()

In [ ]:
# %% [markdown]
# ## Cell 7: Testing Script Using the Modular Components
# This test script uses the above modules to simulate a multi-turn conversation.

# %% [code]
%%writefile test_script.py
"""
Test Script: test_script.py

This script demonstrates:
- Loading the trained model.
- Using the modular MCP and contextual memory.
- Generating responses with dynamic context and safety checks.
"""

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from mcp_module import generate_mcp_context
from context_memory import update_conversation_history, get_conversation_summary

# Load model and tokenizer (adjust path if necessary).
model = AutoModelForCausalLM.from_pretrained("./mistral_mental_health")
tokenizer = AutoTokenizer.from_pretrained("./mistral_mental_health")

def generate_response(prompt: str, session_id: str, user_emotion: str,
                      emotional_trend: str, custom_mcps: str) -> str:
    # Update conversation history with the new user prompt.
    update_conversation_history("User", prompt)
    # Generate conversation summary.
    conversation_summary = get_conversation_summary()
    # Create the full MCP context.
    mcp_context = generate_mcp_context(
        session_id, user_emotion, conversation_summary, emotional_trend, custom_mcps, prompt
    )
    # Combine MCP context with user prompt.
    full_prompt = mcp_context + "\n\nUser: " + prompt
    inputs = tokenizer(full_prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_length=512, num_return_sequences=1)
    bot_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Update conversation history with bot's response.
    update_conversation_history("Bot", bot_response)
    return bot_response

# Example parameters.
session_id = "session_001"
user_emotion = "anxious"
emotional_trend = "worsening"
custom_mcps = "{}"  # Extend if needed.

# Simulated multi-turn conversation.
test_prompts = [
    "Hi, I'm feeling overwhelmed by my work and personal life.",
    "Sometimes I even have thoughts of self-harm.",
    "I don't know how much longer I can go on like this.",
    "What should I do when these feelings take over my life?"
]

for prompt in test_prompts:
    print(f"\nUser: {prompt}")
    response = generate_response(prompt, session_id, user_emotion, emotional_trend, custom_mcps)
    print("Bot:", response)


In [ ]:
# %% [markdown]
# ## Cell 8: Final Checks and Local Testing
# This cell performs a quick local test and creates a ZIP file (for Colab) of the saved model.

import os
print("\nChecking saved files in the output directory:")
print(os.listdir(output_path))

# Quick local test of the saved model.
model = AutoModelForCausalLM.from_pretrained("./mistral_mental_health")
tokenizer = AutoTokenizer.from_pretrained("./mistral_mental_health")
test_prompt = "How can I help someone with anxiety?"
inputs = tokenizer(f"[INST]{test_prompt}[/INST]", return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_length=512)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"\nLocal test response: {response}")

# For Colab: Create a ZIP file of the model for download.
!zip -r mistral_mental_health.zip ./mistral_mental_health
from google.colab import files
files.download('mistral_mental_health.zip')